In [4]:
# ============================================
# 0. 导入依赖（本段代码需要的库）
# ============================================
import pandas as pd                                   # 读写 CSV、处理表格数据
from sklearn.model_selection import train_test_split  # 划分训练集 / 验证集
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF 向量化
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score  # 三个评估指标

from sklearn.linear_model import LogisticRegression   # 逻辑回归分类器
from sklearn.svm import LinearSVC                     # 线性 SVM（不直接输出概率）
from sklearn.calibration import CalibratedClassifierCV  # 概率校准（让 LinearSVC 输出概率）
from sklearn.naive_bayes import MultinomialNB         # 多项式朴素贝叶斯（适合词频特征）
from sklearn.ensemble import RandomForestClassifier   # 随机森林（树模型集成）


# ============================================
# 1. 读入数据并划分训练 / 验证集
# ============================================

train_df = pd.read_csv("train.csv")   # 读取训练集：包含 id、text、label
atest_df = pd.read_csv("Atest.csv")   # 读取测试集A：包含 id、text（无 label）

# 9:1 划分：从训练集中划出 10% 作为验证集
# stratify=train_df["label"] 表示“分层抽样”，保证训练/验证中正负样本比例与原始一致
# random_state 固定随机种子，保证每次划分结果一致（方便复现）
train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

# 从 DataFrame 中取出文本列（X）与标签列（y）
# astype(str) 确保文本类型一致，避免出现 NaN 或非字符串导致向量化报错
X_train_text = train_df["text"].astype(str)   # 训练集文本
y_train      = train_df["label"].astype(int)  # 训练集标签（0/1）

X_val_text   = val_df["text"].astype(str)     # 验证集文本
y_val        = val_df["label"].astype(int)    # 验证集标签

X_atest_text = atest_df["text"].astype(str)   # 测试集文本（最终要输出概率提交）


# ============================================
# 2. 构建 TF-IDF 特征（字符级 n-gram，更适合中文）
# ============================================

# TF-IDF 的作用：把文本转成向量（高维稀疏特征），便于传统 ML 分类器使用
# analyzer="char" 表示按“字符”切分（中文常用），而不是按英文单词空格切分
# ngram_range=(1, 2) 表示用 1-gram（单字） + 2-gram（相邻两字）特征
# min_df=2：某个字符/二元组必须至少在 2 条样本中出现过才保留（过滤极罕见噪声）
# max_features=20000：限制特征维度上限（控制内存与训练速度）
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(1, 2),   # 1-gram + 2-gram
    min_df=2,             # 至少在 2 个样本中出现过
    max_features=20000    # 特征维度上限（可尝试 10000/30000 等）
)

# 只在训练集上 fit（学习词表与 IDF 权重），再 transform 验证/测试
# 这样可以避免“数据泄漏”（否则验证/测试的信息会进入特征构建）
X_train_tfidf = vectorizer.fit_transform(X_train_text)  # 训练集 TF-IDF 稀疏矩阵
X_val_tfidf   = vectorizer.transform(X_val_text)        # 验证集 TF-IDF（用同一词表）
X_atest_tfidf = vectorizer.transform(X_atest_text)      # 测试集 TF-IDF（用同一词表）


# ============================================
# 3. 封装一个通用的评估 + 生成提交文件函数
# ============================================
def evaluate_and_submit(clf, model_name, use_calibrated_proba=False):
    """
    作用：对一个 sklearn 分类器完成以下流程：
    1) 在训练集上 fit
    2) 在验证集上输出 Accuracy / F1 / AUC
    3) 在测试集 A 上输出正类概率 prob，并保存为提交文件

    参数说明：
    - clf: 一个 sklearn 模型对象（尚未 fit）
    - model_name: 保存文件名时使用（如 "logreg_tfidf"）
    - use_calibrated_proba:
        若模型本身不支持 predict_proba（例如 LinearSVC），
        则用 CalibratedClassifierCV 包一层做概率校准，以获得可用于 AUC 的概率输出。
    """

    # 如果需要校准概率（主要针对 LinearSVC 这种没有 predict_proba 的模型）
    if use_calibrated_proba:
        # CalibratedClassifierCV 会在内部做交叉验证拟合校准器
        # method="sigmoid" 常用且稳定（Platt Scaling）
        clf = CalibratedClassifierCV(clf, cv=3, method="sigmoid")

    # -------------------------
    # (1) 训练：在训练集 TF-IDF 特征上拟合模型
    # -------------------------
    clf.fit(X_train_tfidf, y_train)

    # -------------------------
    # (2) 验证集预测类别（用于 Accuracy、F1）
    # -------------------------
    val_pred = clf.predict(X_val_tfidf)  # 输出 0/1

    # -------------------------
    # (3) 验证集预测正类概率（用于 AUC）
    # -------------------------
    if hasattr(clf, "predict_proba"):
        # predict_proba 输出形状 [N, 2]，取第 2 列即正类(1)概率
        val_proba = clf.predict_proba(X_val_tfidf)[:, 1]
    else:
        # 理论上不会走到这里：LinearSVC 已经被 Calibrated 包装过
        raise RuntimeError(f"{model_name} 不支持 predict_proba")

    # -------------------------
    # (4) 计算指标
    # -------------------------
    acc = accuracy_score(y_val, val_pred)      # 准确率：预测正确比例
    f1  = f1_score(y_val, val_pred)            # F1：precision/recall 的调和平均
    auc = roc_auc_score(y_val, val_proba)      # AUC：排序能力（线上主要指标）

    # 打印结果，便于你记录在实验报告中
    print(f"{model_name} 验证集表现：")
    print(f"  Accuracy = {acc:.4f}")
    print(f"  F1       = {f1:.4f}")
    print(f"  AUC      = {auc:.4f}")
    print("-" * 40)

    # -------------------------
    # (5) 在测试集 A 上预测正类概率并生成提交文件
    # -------------------------
    atest_proba = clf.predict_proba(X_atest_tfidf)[:, 1]  # 测试集正类概率

    # 组装提交格式：两列（id, prob）
    submission_df = pd.DataFrame({
        "id": atest_df["id"],     # 测试集样本 id
        "prob": atest_proba       # 预测为“假新闻(label=1)”的概率
    })

    # 保存为 csv：文件名里写上模型名称，便于后续集成
    save_name = f"{model_name}_submission_a.csv"
    submission_df.to_csv(save_name, index=False, encoding="utf-8")
    print(f"测试集提交文件已保存：{save_name}\n")

    # 返回指标，方便你后面做表格汇总或用于加权集成
    return acc, f1, auc


# ============================================
# 4. 模型 1：TF-IDF + 逻辑回归
# ============================================

# 逻辑回归是线性分类器，适合高维稀疏 TF-IDF 特征，并且天然输出概率（对 AUC 友好）
logreg_clf = LogisticRegression(
    C=4.0,                # 正则强度的倒数：C 越大正则越弱，模型更“拟合”（可调参）
    max_iter=200,          # 最大迭代次数，防止不收敛
    solver="liblinear",    # liblinear 对稀疏特征常用且稳定（适合二分类）
    n_jobs=-1              # 并行（注意：liblinear 有时不完全支持 n_jobs，影响不大）
)

# 训练 + 验证 + 保存提交
logreg_metrics = evaluate_and_submit(logreg_clf, "logreg_tfidf")


# ============================================
# 5. 模型 2：TF-IDF + 线性 SVM（LinearSVC + 概率校准）
# ============================================

# LinearSVC 是线性 SVM，常在文本分类上很强，但默认不输出概率
svm_clf = LinearSVC(
    C=1.0,                # 惩罚系数：越大越倾向把训练集分对（可能过拟合），可调参
    max_iter=2000         # 最大迭代次数
)

# use_calibrated_proba=True：用 CalibratedClassifierCV 包装，得到 predict_proba
svm_metrics = evaluate_and_submit(svm_clf, "linearsvc_tfidf", use_calibrated_proba=True)


# ============================================
# 6. 模型 3：TF-IDF + 多项式朴素贝叶斯
# ============================================

# 朴素贝叶斯适合词频/计数类特征，在文本任务中是经典 baseline，速度很快
nb_clf = MultinomialNB(
    alpha=0.5    # 拉普拉斯平滑系数：防止概率为 0；可试 0.1 / 0.5 / 1.0
)

# 训练 + 验证 + 保存提交
nb_metrics = evaluate_and_submit(nb_clf, "mnb_tfidf")


# ============================================
# 7. 模型 4：TF-IDF + 随机森林
# ============================================

# 随机森林是树模型集成，能拟合非线性关系；
# 但在高维稀疏文本特征上不一定占优（可作为对比实验）
rf_clf = RandomForestClassifier(
    n_estimators=300,     # 树的数量（越多越稳但更慢）
    max_depth=None,       # 不限制深度，让树自己生长（可能过拟合，可尝试限制深度）
    n_jobs=-1,            # 并行加速
    random_state=42       # 固定随机种子，便于复现
)

# 训练 + 验证 + 保存提交
rf_metrics = evaluate_and_submit(rf_clf, "rf_tfidf")


D:\miniconda\lib\site-packages\sklearn\linear_model\_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 32.
  warnings.warn(


logreg_tfidf 验证集表现：
  Accuracy = 0.9293
  F1       = 0.9249
  AUC      = 0.9795
----------------------------------------
测试集提交文件已保存：logreg_tfidf_submission_a.csv

linearsvc_tfidf 验证集表现：
  Accuracy = 0.9266
  F1       = 0.9222
  AUC      = 0.9823
----------------------------------------
测试集提交文件已保存：linearsvc_tfidf_submission_a.csv

mnb_tfidf 验证集表现：
  Accuracy = 0.8804
  F1       = 0.8590
  AUC      = 0.9603
----------------------------------------
测试集提交文件已保存：mnb_tfidf_submission_a.csv

rf_tfidf 验证集表现：
  Accuracy = 0.8886
  F1       = 0.8761
  AUC      = 0.9691
----------------------------------------
测试集提交文件已保存：rf_tfidf_submission_a.csv

